In [ ]:
import pandas as pd
import json
import numpy as np
import requests
from tqdm.auto import tqdm
from config import URL, TOKEN


import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
headers = {
    "Authorization": f"Bearer {TOKEN}"
}

itemCount = np.inf
data = {"entries": []}

offset = 0
steps = 1000

progress = None

while len(data["entries"]) < itemCount:

    response = requests.get(
        f"{URL}person/caregiver/query?filter=section.eingestellt-am:answerDateTime.timeUtc:gte:20170101000000,section.eingestellt-am:answerDateTime.timeUtc:lt:20260101000000&limit={steps}&skip={offset}",
        headers=headers,
        verify=False
    )

    if response.status_code == 200:
        responseData = response.json()

        entries = responseData["entries"]

        # Total becomes known after first request
        if progress is None:
            itemCount = responseData["pagingInfo"]["itemCount"]

            progress = tqdm(
                total=itemCount,
                desc="Downloading caregivers",
                unit="entries"                
            )

        data["entries"].extend(entries)

        # Advance by actual number received
        progress.update(len(entries))

        offset += steps

    else:
        print(response.json())
        break

if progress is not None:
    progress.close()

In [ ]:
exceptions = [
	"metadata",
	"locality",
	"postalcode",
	"contactoptions",
	"personFullName",
	"personFullNameNoTitle",
	"tenantid",
	"persontype",
	"adresse",
	"mailAddressing",
	"socialInsuranceNumber",
	"comments",
	"givenName",
	"familyName",
	"primaryEmailAddress",
	'addresses',
	'primaryPhoneNumber',
	'address',
	'residentialAddress',
	'address',
	'geoLocation',
	'billingAddress',
	'serviceAddress',
	'legalAddress',
	'businessAddress',
	'emailAddresses',
	'phoneNumbers',
	'personFullNameNoTitle',
	'personFullName',
	'mailAddressing',
	'postalAddress',
	'contractualAddressing',
	'caatsDateOfBirth',
	'gender',
	'locations',
	'bankDetails',
	'personStatus',
	"chapterId",
	"language",
	"ordinal",
	"sectionId",
	"academicTitlePrefix",
	"personNr",
	"content",
	"consecutiveNumber",
	'Klient:innen_Empfohlen_Ja',
	'Klient:innen_Erstrkontakt_erfassen_Ja',
	'Klient:innen_Bew_Ein_Ja',
	'abrech-akonto',
	'abrech-buerge-hinterlegt',
	'pers-visite-keine',
	'medizinische-delegation-hochgeladen',
	'medizinische-delegation-nicht-notwendig',
	'Klient:innen_Medizinische_Delegation_Ja',
	'pflegerische-delegation-hochgeladen',
	'pflegerische-delegation-nicht-notwendig',
	'Klient:innen_Pflegerische_Delegation_JA',
	"admin-beruf",
	'pflegevisite-letzte-date',
	'pflegevisite-letzte-dgkp',
	'pflegerisch-person',
	'erstkontakt-bemerkung',
	"admin-kooperationspartner",
	'birthName',
'confessionId',
'confession',
'nationalityId',
'placeOfBirth',
'paymentBlockReason',
'chamberOfCommerceMembershipNr',
'avatarFileId',
'empfohlen-von-category',
'visibilityConditionId',
'deutschkennt-bew-von',
'Kinder_Nein',
'covid-1',
'covid-2',
'covid-3',
"cpvod-1-mit",
"covid-1-wo",
"cpvod-2-mit",
"covid-2-wo",
"cpvod-3-mit",
"covid-3-wo",
"gewerbeschein_nr",

"PFSystem",
'keine-kurse',
'agentur-zuletzt',
'BE_PERSONALAUSWEIS_NR',
'BE_PERSONALAUSWEIS_BEHOERDE',
'BE_PERSONALAUSWEIS_GUELTIG_AB',
'BE_PERSONALAUSWEIS'
]

exceptions = [x.lower() for x in exceptions]

In [ ]:
def extractStatement(statement):
    field = {}
    if statement["statementId"].lower() in exceptions:
        return field

    match statement["answerScheme"]:
        case 1:
            field[statement["statementId"]] = statement["answerValue"]["displayText"]
        case 2:
            #do nothing
            field = {}
        case 3:
            field[statement["statementId"]] = statement["answerYesno"] == 2
        case 4:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False

        case 5:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False
        case 8:
            if "answerDateTime" in statement.keys():
                if "userLocalTime" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["userLocalTime"]
                elif "timeUtc" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["timeUtc"]
        case _:
            print(f'Found statement: {statement["answerScheme"]} - {statement["statementId"]} -  {statement}')


    return field
        

def returnEntry(entry):
    keys = entry.keys()
    finalFields = {}
    if "statementId" in keys:
        finalFields = finalFields | extractStatement(entry)
    else:
        for key in keys:
            
            if key.lower() in exceptions:
                continue
            
            typing = type(entry[key]).__name__
            match typing:
                case "str":
                    finalFields[key] = entry[key]
                    
                case "int":
                    finalFields[key] = entry[key]

                case "bool":
                    finalFields[key] = entry[key]

                case "float":
                    finalFields[key] = entry[key]
                    
                case "dict":
                    if "displayText" in entry[key].keys():
                        finalFields[key] = entry[key]["displayText"]
                        continue
                    else:
                        finalFields = finalFields | returnEntry(entry[key])
                    
                case "list":
                    for item in entry[key]:
                        if type(item).__name__ == "dict":
                            finalFields = finalFields | returnEntry(item)
                        

                case _:
                    print(typing)


    
    return finalFields

In [ ]:

totalData = []

if data["entries"] is not None:
    for entry in data["entries"]:
        test = returnEntry(entry)
        totalData.append(test)
        
df = pd.DataFrame(totalData)
df

In [ ]:
for col in df.columns:
    non_null = df[col].dropna()

    if len(non_null) > 0 and non_null.isin([True, False]).all():
        df[col] = df[col].fillna(False).astype(bool)

bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns

df[bool_cols] = df[bool_cols].fillna(False)


df = df.replace(r'^\s*$', np.nan, regex=True)

df.to_csv("caregiverRaw.csv", index=False, sep=";")

df

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
protected_cols = ["eingestellt-am", "id", "ausgesch-am"]

# All normal feature columns
feature_cols = df.columns.drop(protected_cols)

# Keep only feature columns with <= 10% missing
keep_features = feature_cols[
    df[feature_cols].isna().mean() <= 0.10
]

# Keep protected columns + surviving features
df = df[protected_cols + keep_features.tolist()]

# Drop rows with missing values in:
# - eingestellt-am
# - id
# - any remaining feature
#
# ausgesch-am is intentionally excluded
required_cols = ["eingestellt-am", "id"] + keep_features.tolist()

df = df.dropna(subset=required_cols)

In [ ]:
df

In [ ]:
min_unique = 2
max_unique = len(df) * 0.9


n_unique = df.nunique()

cols_to_drop = n_unique[
    (n_unique < min_unique) | (n_unique > max_unique)
].index

cols_to_drop = [col for col in cols_to_drop if col not in ["id", "ausgesch-am", "eingestellt-am"]]

df = df.drop(columns=cols_to_drop)

df

In [ ]:
if "genderId" in df.columns:
    df = pd.get_dummies(df, columns=["genderId"], prefix="gender", drop_first=True)
if "orgRef" in df.columns:
    df = pd.get_dummies(df, columns=["orgRef"], prefix="org", drop_first=True)
if "nationality" in df.columns:
    df = pd.get_dummies(df, columns=["nationality"], prefix="nationality", drop_first=True)
if "preferredTransport" in df.columns:
    df = pd.get_dummies(df, columns=["preferredTransport"], prefix="preferredTransport", drop_first=True)
if "pers-dat-sprach-mutt" in df.columns:
    df = pd.get_dummies(df, columns=["pers-dat-sprach-mutt"], prefix="mutSprach", drop_first=True)
if "admin-betr-gsber" in df.columns:
    df = pd.get_dummies(df, columns=["admin-betr-gsber"], prefix="geschBereich", drop_first=True)
if "pflegestatus-paket" in df.columns:
    df = pd.get_dummies(df, columns=["pflegestatus-paket"], prefix="pflegestatus", drop_first=True)

if "PFKompSprache" in df.columns:
	df["PFKompSprache"] = df["PFKompSprache"].astype(float).round(0).astype(int)
if "PFKompHaushalt" in df.columns:
	df["PFKompHaushalt"] = df["PFKompHaushalt"].astype(float).round(0).astype(int)
if "PFKompBetreuung" in df.columns:
	df["PFKompBetreuung"] = df["PFKompBetreuung"].astype(float).round(0).astype(int)
if "PFKompEmotional" in df.columns:
	df["PFKompEmotional"] = df["PFKompEmotional"].astype(float).round(0).astype(int)
if "PFKompAllgemein" in df.columns:
	df["PFKompAllgemein"] = df["PFKompAllgemein"].astype(float).round(0).astype(int)
if "eingestellt-am" in df.columns:
	df["eingestellt-am"] = df["eingestellt-am"].astype(str).str[:8].astype(float)
if "ausgesch-am" in df.columns:
	df["ausgesch-am"] = df["ausgesch-am"].astype(str).str[:8].astype(float)
if "pers-koerper-gewicht" in df.columns:
	df["pers-koerper-gewicht"] = df["pers-koerper-gewicht"].astype(float).round(0).astype(int)
if "pers-koerper-groesse" in df.columns:
	df["pers-koerper-groesse"] = df["pers-koerper-groesse"].astype(float).round(0).astype(int)

In [ ]:



cols = df.columns.drop(
    ["id", "ausgesch-am", "eingestellt-am"]
)

converted = df[cols].apply(pd.to_numeric, errors="coerce")

# Columns containing problematic values
problem_cols = cols[
    converted.isna().any() & df[cols].notna().any()
]

print(problem_cols)

if len(problem_cols) ==0:
    numeric = df[cols].apply(
            pd.to_numeric,
            errors="coerce"
    )

        # Keep only rows where all columns converted successfully
    df = df[numeric.notna().all(axis=1)].copy()

        # Convert to int
    df[cols] = df[cols].astype(int)

In [ ]:
df

In [ ]:
correlation_matrix = df.drop(columns=["id", "ausgesch-am", "eingestellt-am"]).corr()

In [ ]:
import matplotlib.pyplot as plt

corr = df.drop(columns=["id"]).corr()

plt.figure(figsize=(12, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout()
plt.show()

In [ ]:
def remove_correlated_features(df, threshold=0.8, exclude=["id"]):

    result = df.copy()
    dropped = []

    while True:

        features = result.drop(
            columns=exclude,
            errors="ignore"
        )

        corr = features.corr().abs()

        # Remove diagonal
        np.fill_diagonal(corr.values, 0)

        # Find strongest remaining correlation
        max_corr = corr.max().max()

        if max_corr <= threshold:
            break

        # Find the pair
        var1, var2 = corr.stack().idxmax()

        # Mean absolute correlation with all other variables
        score1 = corr.loc[var1].mean()
        score2 = corr.loc[var2].mean()

        # Drop the more redundant variable
        if score1 > score2:
            drop = var1
        else:
            drop = var2

        dropped.append({
            "dropped": drop,
            "var1": var1,
            "var2": var2,
            "pair_corr": max_corr,
            "score_var1": score1,
            "score_var2": score2
        })

        result = result.drop(columns=drop)

    return result, pd.DataFrame(dropped)

In [ ]:
df_reduced, dropped = remove_correlated_features(
    df,
    threshold=0.7,
    exclude=["id", "ausgesch-am", "eingestellt-am"]
)

print(dropped)

In [ ]:
df_reduced

In [ ]:
df_reduced.to_csv("caregivers.csv", sep=";", index=False)